# AETHER BFCL official execution

Runs the pinned native official package on development-scoped preparation only.
This notebook does not access AETHER validation/test splits. It exports raw
official outputs plus a hash-verified AETHER wrapper bundle.


In [ ]:
from pathlib import Path
import json,os,re,shutil,subprocess,sys,tarfile,hashlib
WORK=Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()/'aether-execution-output'
WORK.mkdir(parents=True,exist_ok=True)
ATTEMPT=os.environ.get('AETHER_BFCL_ATTEMPT','attempt-1')
assert re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9._-]{0,63}',ATTEMPT), f'Invalid attempt identifier: {ATTEMPT!r}'
RUNTIME=Path('/tmp/aether-bfcl')
REPO=RUNTIME/'aether'; VENV=RUNTIME/'venv'; PY=VENV/'bin'/'python'
OUTPUT=WORK/f'aether-bfcl-official-{ATTEMPT}'; COMMIT='2e922a804ae7b6db255d7f879933c5efab1e22e5'
if OUTPUT.exists(): raise FileExistsError(f'Refusing to repeat or overwrite this attempt: {OUTPUT}')
URL='https://github.com/ama1a005/aether.git'
os.environ.update({'PYTHONNOUSERSITE':'1','MPLBACKEND':'Agg'})
RUNTIME.mkdir(parents=True,exist_ok=True)


In [ ]:
if 'RUNTIME' not in globals() or 'REPO' not in globals() or 'VENV' not in globals():
    raise RuntimeError('Setup variables are missing. Run the notebook setup cell (Cell 1) successfully before this cell.')
if RUNTIME.exists() and (REPO.exists() or VENV.exists()):
    raise RuntimeError(f'Execution paths already exist under {RUNTIME}. Use a fresh Kaggle session; existing files are preserved.')
subprocess.run([sys.executable,'-m','pip','install','--upgrade','uv==0.11.8'],check=True)
UV=shutil.which('uv')
if UV is None:
    raise RuntimeError('Pinned uv==0.11.8 installation succeeded but no uv executable is on PATH.')
def parse_uv_version(output):
    match=re.match(r'^\s*uv\s+(\d+\.\d+\.\d+)(?:\s|$)',output)
    if match is None:
        raise RuntimeError(f'Could not parse uv version from --version output: {output!r}')
    return match.group(1)

uv_output=subprocess.check_output([UV,'--version'],text=True).strip()
uv_version=parse_uv_version(uv_output)
if uv_version!='0.11.8':
    raise RuntimeError(f'Expected uv 0.11.8; found version {uv_version!r} in {uv_output!r}')
env=os.environ.copy(); env['UV_PYTHON_INSTALL_DIR']=str(RUNTIME/'uv-python')
subprocess.run([UV,'venv','--python','3.11.9',str(VENV)],check=True,env=env)
if not PY.is_file():
    raise RuntimeError(f'uv did not create the expected Python executable: {PY}')
subprocess.run(['git','clone','--branch','lm','--single-branch',URL,str(REPO)],check=True)
subprocess.run(['git','fetch','origin','m1/target-harness'],cwd=REPO,check=True)
subprocess.run(['git','checkout','--detach',COMMIT],cwd=REPO,check=True)
actual_commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
if actual_commit!=COMMIT:
    raise RuntimeError(f'Checked out {actual_commit}, expected pinned commit {COMMIT}')
if subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip():
    raise RuntimeError('Pinned source checkout is not clean; refusing dependency setup.')
SETUP_COMPLETE=True


In [ ]:
if not globals().get('SETUP_COMPLETE',False):
    raise RuntimeError('Repository setup is incomplete. Run the preceding setup cell successfully before installing dependencies.')
if not REPO.is_dir() or not PY.is_file() or 'env' not in globals():
    raise RuntimeError('Repository, pinned Python, or environment is missing. Restart with a fresh Kaggle session and run cells in order.')
subprocess.run([UV,'pip','install','--python',str(PY),'-r',str(REPO/'requirements-lock.txt')],check=True,env=env)
import hashlib
WHEELHOUSE=RUNTIME/'wheelhouse'; WHEELHOUSE.mkdir(parents=True,exist_ok=True)
subprocess.run([str(PY),'-m','pip','download','--no-deps','--only-binary=:all:','--dest',str(WHEELHOUSE),'bfcl_eval==2026.3.23'],check=True)
WHEELS=sorted(WHEELHOUSE.glob('bfcl_eval-2026.3.23-*.whl'))
assert len(WHEELS)==1, f'Expected exactly one pinned BFCL wheel, found {len(WHEELS)}'
WHEEL=WHEELS[0]; WHEEL_SHA256=hashlib.sha256(WHEEL.read_bytes()).hexdigest()
assert WHEEL_SHA256=='3bb6dfa5f0c68ad403c9ec50b00db2bb3b4cc9b38ab1ff33f48fe30d853d3a0a', WHEEL_SHA256
subprocess.run([str(PY),'-m','pip','install',str(WHEEL)],check=True)
subprocess.run([UV,'pip','install','--python',str(PY),'-r',str(REPO/'requirements-bfcl-vllm.txt'),'--torch-backend','cu130'],check=True,env=env)
assert subprocess.check_output([str(PY),'-c','import torch; from importlib.metadata import version; print(version("vllm"),torch.__version__,torch.version.cuda)'],text=True).strip().startswith('0.34.0 2.14.0+ 13.0')

subprocess.run([str(PY),'-m','pytest','tests/unit/test_external_bundle.py','tests/unit/test_native_benchmark_runners.py','tests/unit/test_native_official_package.py','-q'],cwd=REPO,check=True)
DEPENDENCIES_COMPLETE=True


In [ ]:
if not globals().get('SETUP_COMPLETE',False):
    raise RuntimeError('Repository setup is incomplete. Run all preceding setup cells successfully in order.')
if not globals().get('DEPENDENCIES_COMPLETE',False):
    raise RuntimeError('Dependency installation or model-free tests did not complete. Fix the preceding cell before continuing.')
if not REPO.is_dir() or not PY.is_file() or 'env' not in globals():
    raise RuntimeError('Repository, pinned Python, or environment is missing. Restart with a fresh Kaggle session and run cells in order.')
import sys
sys.path.insert(0,str(REPO))
if Path.cwd()!=REPO:
    os.chdir(REPO)
actual_commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
if actual_commit!=COMMIT:
    raise RuntimeError(f'Expected pinned repository commit {COMMIT}, found {actual_commit}')
from huggingface_hub import snapshot_download
from benchmarks.run_native_official import _load_config
from benchmarks.nov19_bfcl.preflight import check_backend_and_hardware, check_package
from causal.remediation_freeze import frozen_text_hash_matches
from experiments.external_bundle import verify_protocol_approval
CONFIG=REPO/'configs/benchmarks/phase1_external.yaml'
_load_config(CONFIG,'bfcl')
APPROVAL=REPO/'results/phase1/protocol_approvals/bfcl_official.json'
verify_protocol_approval(CONFIG,APPROVAL)
wheel_ok,wheel_reasons,_=check_package(wheel_path=WHEEL)
gpu_ok,gpu_reasons,gpu_details=check_backend_and_hardware('vllm',1)
assert wheel_ok, wheel_reasons
assert gpu_ok, {'reasons':gpu_reasons,'details':gpu_details}
FROZEN_MANIFEST=REPO/'results/phase1/model_qualification/qwen15b_kaggle_20261110/model_snapshot_manifest.json'
assert frozen_text_hash_matches(FROZEN_MANIFEST,'1b282b730f38fbe992608e8837fe0f619463a767f93e08d97cd0950d55644f11')
frozen=json.loads(FROZEN_MANIFEST.read_text(encoding='utf-8'))
REVISION='989aa7980e4cf806f80c7fef2b1adb7bc71aa306'
snapshot=Path(snapshot_download('Qwen/Qwen2.5-1.5B-Instruct',revision=REVISION,cache_dir=RUNTIME/'hf-cache',max_workers=1))
expected={item['path']:(item['size'],item['sha256']) for item in frozen['files']}
observed={p.relative_to(snapshot).as_posix():(p.stat().st_size,hashlib.sha256(p.read_bytes()).hexdigest()) for p in sorted(snapshot.rglob('*')) if p.is_file()}
assert observed==expected, 'Downloaded snapshot differs from the independently retained qualification manifest'
SNAPSHOT_MANIFEST=WORK/'model_snapshot_manifest.json'
SNAPSHOT_MANIFEST.write_text(json.dumps({'model_id':'Qwen/Qwen2.5-1.5B-Instruct','resolved_revision':REVISION,'tokenizer_revision':REVISION,'snapshot_path':str(snapshot),'files':[{'path':name,'size':size,'sha256':digest} for name,(size,digest) in sorted(observed.items())]},indent=2),encoding='utf-8')
PREFLIGHT_COMPLETE=True


In [ ]:
if not globals().get('PREFLIGHT_COMPLETE',False):
    raise RuntimeError('Required setup/preflight cells did not complete. Run all preceding cells in order.')
command=[str(PY),'-m','benchmarks.run_native_official','bfcl','--config',str(REPO/'configs/benchmarks/phase1_external.yaml'),'--output',str(OUTPUT)]
command += ['--snapshot-manifest',str(SNAPSHOT_MANIFEST),'--package-wheel',str(WHEEL),'--backend','vllm','--num-gpus','1']
subprocess.run(command,cwd=REPO,check=True)
archive=WORK/f'aether-bfcl-official-{ATTEMPT}-results.tar.gz'
with tarfile.open(archive,'w:gz') as bundle: bundle.add(OUTPUT,arcname=OUTPUT.name)
print(archive,hashlib.sha256(archive.read_bytes()).hexdigest())
subprocess.run([str(PY),'-m','scripts.verify_external_bundle',str(OUTPUT),'--expected-kind','bfcl_official'],cwd=REPO,check=True)
bfcl_summary=json.loads((OUTPUT/'summary.json').read_text(encoding='utf-8'))
if bfcl_summary.get('overall')=='PASS':
    subprocess.run([str(PY),'-m','scripts.verify_bfcl_bundle',str(OUTPUT)],cwd=REPO,check=True)
else:
    print('Retaining a blocked/failed BFCL attempt; this is not a score.')

